[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CS7150/classdemos/blob/main/optimization/ema.ipynb)

# EMA and bias correction: three ways to average a stream

Suppose you're watching a stream of numbers $x_1, x_2, x_3, \dots$ arrive one at a time — gradients during training, sensor readings, anything noisy — and at every step $t$ you want a single running estimate of "the current level" of that stream. There are three natural ways to define that running estimate, and they trade off adaptivity against a subtle startup bias.

**1. The plain running average.** The simplest choice treats every sample equally, no matter how old:

$$\mathrm{Avg}_t = \frac{1}{t}\sum_{i=1}^{t} x_i,$$

which can be computed incrementally, without storing the whole history, as

$$\mathrm{Avg}_t \leftarrow \mathrm{Avg}_{t-1} + \frac{x_t - \mathrm{Avg}_{t-1}}{t}.$$

This is unbiased and exact, but it has a long memory: after 1000 samples, sample number 1 still carries the same weight ($1/1000$) as sample number 1000. If the underlying signal drifts or jumps, $\mathrm{Avg}_t$ adapts very slowly, because it can never "forget" the accumulated past.

**2. The exponential moving average (EMA).** To adapt faster, we can instead give recent samples more weight and let old ones decay away geometrically, using a decay rate $\beta \in (0,1)$ (close to 1 for slow decay, close to 0 for fast decay):

$$\mathrm{EMA}_t \leftarrow \beta\,\mathrm{EMA}_{t-1} + (1-\beta)\,x_t, \qquad \mathrm{EMA}_0 = 0.$$

Unrolling the recurrence shows explicitly that it's a weighted average with geometrically decaying weights $(1-\beta)\beta^k$ on the sample from $k$ steps back:

$$\mathrm{EMA}_t = (1-\beta)\sum_{k=0}^{t-1}\beta^k\,x_{t-k}.$$

This adapts to change much faster than the plain average, since old samples' weight shrinks exponentially instead of just as $1/t$. But there's a catch: because $\mathrm{EMA}_0$ is initialized to $0$, the very first few updates are still mostly "made of" that fictitious zero, not of real data. If the true signal starts away from zero, $\mathrm{EMA}_t$ will *underestimate* it for a while — this is the **cold-start bias**.

**3. The bias-corrected EMA.** We can quantify the cold-start bias exactly. Since $\mathrm{EMA}_t$ is built entirely from a mixture of the true samples $x_1,\dots,x_t$ and the initial $\mathrm{EMA}_0=0$, the total weight actually resting on real data is exactly $1-\beta^t$ (the geometric weights above sum to this, not to $1$). So dividing by that missing fraction corrects the estimate back to the right scale:

$$\widehat{\mathrm{EMA}}_t = \frac{\mathrm{EMA}_t}{1-\beta^t}.$$

At $t=1$ this makes $\widehat{\mathrm{EMA}}_1 = x_1$ exactly — no bias at all — and as $t\to\infty$, $\beta^t\to 0$ so the correction factor $1/(1-\beta^t)\to 1$ and $\widehat{\mathrm{EMA}}_t$ smoothly converges back to the plain (uncorrected) $\mathrm{EMA}_t$. This is exactly the bias-correction trick used for the first and second moment estimates inside the Adam optimizer.

This notebook reproduces the same three curves as the
[EMA demo page](https://cs7150.github.io/classdemos/demos/optimizers/ema.html),
with the core recurrences above written as plain, tiny Python loops.

In [ ]:
#@title Setup: data generator + plotting helper (double-click to inspect) { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def make_stream(kind="step", n=44, seed=0):
    """Synthetic data streams, all starting away from zero so the EMA's
    cold-start bias (and the correction fixing it) is visible immediately."""
    rng = np.random.RandomState(seed)
    xs = np.zeros(n)
    if kind == "step":
        xs[:] = -0.9
        xs[:22] = 1.3
    elif kind == "sine":
        t = np.arange(n)
        xs = np.cos(t * 0.35) + rng.uniform(-1, 1, n) * 0.25
    elif kind == "walk":
        xs[0] = 1.1
        for t in range(1, n):
            xs[t] = xs[t - 1] + rng.uniform(-1, 1) * 0.3
    else:
        raise ValueError(kind)
    return xs

def plot_ema(xs, avg, ema_raw, ema_hat, beta, title):
    t = np.arange(1, len(xs) + 1)
    plt.figure(figsize=(8, 4))
    plt.scatter(t, xs, color="0.6", s=14, label="raw data $x_t$", zorder=3)
    plt.plot(t, avg, color="tab:green", linestyle=(0, (2, 3)), lw=1.8, label="equal average forever")
    plt.plot(t, ema_raw, color="tab:orange", linestyle=(0, (5, 3)), lw=1.8, label="raw EMA")
    plt.plot(t, ema_hat, color="tab:blue", lw=2.4, label="bias-corrected EMA")
    plt.axhline(0, color="0.85", lw=1, zorder=0)
    plt.title(f"{title} (β = {beta})")
    plt.xlabel("t")
    plt.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.show()


## The three running statistics

Each one is a single-line update applied once per incoming sample `x`:

- **Equal average forever:** `avg = avg + (x - avg) / t` — every sample gets weight `1/t`, so it never forgets.
- **Raw EMA:** `ema_raw = beta*ema_raw + (1-beta)*x`, started at `ema_raw = 0` — recent samples dominate, but the `0` start biases early estimates toward zero.
- **Bias-corrected EMA:** `ema_hat = ema_raw / (1 - beta**t)` — dividing out exactly the fraction of weight still sitting on that fictitious zero start.

In [ ]:
beta = 0.90
xs = make_stream("step", n=44)

avg, ema_raw, ema_hat = [], [], []
running_avg = 0.0
running_ema = 0.0
t = 0

for x in xs:
    t += 1
    running_avg = running_avg + (x - running_avg) / t     # equal average forever
    running_ema = beta * running_ema + (1 - beta) * x      # raw EMA (starts biased toward 0)
    corrected = running_ema / (1 - beta**t)                # bias-corrected EMA

    avg.append(running_avg)
    ema_raw.append(running_ema)
    ema_hat.append(corrected)

plot_ema(xs, avg, ema_raw, ema_hat, beta, "Step stream")


## Watch the cold-start bias directly

Right after the first sample, the raw EMA has barely moved off 0 while the
bias-corrected EMA jumps straight to (roughly) the true value. Try lowering
`beta` (faster decay, less bias but noisier) or raising it (slower, smoother,
more pronounced early bias) on the other streams below.

In [ ]:
for kind in ["step", "sine", "walk"]:
    xs = make_stream(kind, n=44, seed=1)
    avg, ema_raw, ema_hat = [], [], []
    running_avg = running_ema = 0.0
    t = 0
    for x in xs:
        t += 1
        running_avg = running_avg + (x - running_avg) / t
        running_ema = beta * running_ema + (1 - beta) * x
        avg.append(running_avg)
        ema_raw.append(running_ema)
        ema_hat.append(running_ema / (1 - beta**t))
    plot_ema(xs, avg, ema_raw, ema_hat, beta, kind)
